T5 stands for Text-to-Text Transfer Transformer. The core idea is dead simple — every NLP task gets framed as "text in, text out."

In [2]:
from transformers import T5Tokenizer, T5ForConditionalGeneration # Import the tokenizer and the text generator

# Select the t5 small model
tokenizer = T5Tokenizer.from_pretrained('t5-small')
model = T5ForConditionalGeneration.from_pretrained('t5-small')

text = """summarize: The quick brown fox jumped over the lazy dog. The dog was sleeping peacefully in the garden when the fox
appeared from the nearby forest. The fox was known for being particularly agile and mischievous.
After jumping over the dog, the fox continued running through the garden and disappeared into the bushes on the other side."""

inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True) # text to summarize
outputs = model.generate(inputs['input_ids'], max_length=50, output_scores=True, return_dict_in_generate=True) # generates summary in tokens
summary = tokenizer.decode(outputs[0], skip_special_tokens=True) # transforms tokens into words from the embedding table
print(summary)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

['the quick brown fox jumped over the lazy dog . the fox was known for being particularly agile and mischievous .']


In [22]:
tokenizer.convert_ids_to_tokens(outputs.sequences[0])

['<pad>',
 '▁the',
 '▁quick',
 '▁brown',
 '▁',
 'fox',
 '▁',
 'jumped',
 '▁over',
 '▁the',
 '▁lazy',
 '▁dog',
 '▁',
 '.',
 '▁the',
 '▁',
 'fox',
 '▁was',
 '▁known',
 '▁for',
 '▁being',
 '▁particularly',
 '▁agile',
 '▁and',
 '▁mis',
 'chie',
 'vous',
 '▁',
 '.',
 '</s>']

In [3]:
# Try a longer input text
from datasets import load_dataset
ds = load_dataset("xsum", split="test[:5]")
print(ds[0]['document'])

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/300M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/16.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/204045 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11332 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11334 [00:00<?, ? examples/s]

Prison Link Cymru had 1,099 referrals in 2015-16 and said some ex-offenders were living rough for up to a year before finding suitable accommodation.
Workers at the charity claim investment in housing would be cheaper than jailing homeless repeat offenders.
The Welsh Government said more people than ever were getting help to address housing problems.
Changes to the Housing Act in Wales, introduced in 2015, removed the right for prison leavers to be given priority for accommodation.
Prison Link Cymru, which helps people find accommodation after their release, said things were generally good for women because issues such as children or domestic violence were now considered.
However, the same could not be said for men, the charity said, because issues which often affect them, such as post traumatic stress disorder or drug dependency, were often viewed as less of a priority.
Andrew Stevens, who works in Welsh prisons trying to secure housing for prison leavers, said the need for accommodat

In [5]:
input = 'summarize: ' + ds[0]['document'] # Add a summarize command in the beginning
input

'summarize: Prison Link Cymru had 1,099 referrals in 2015-16 and said some ex-offenders were living rough for up to a year before finding suitable accommodation.\nWorkers at the charity claim investment in housing would be cheaper than jailing homeless repeat offenders.\nThe Welsh Government said more people than ever were getting help to address housing problems.\nChanges to the Housing Act in Wales, introduced in 2015, removed the right for prison leavers to be given priority for accommodation.\nPrison Link Cymru, which helps people find accommodation after their release, said things were generally good for women because issues such as children or domestic violence were now considered.\nHowever, the same could not be said for men, the charity said, because issues which often affect them, such as post traumatic stress disorder or drug dependency, were often viewed as less of a priority.\nAndrew Stevens, who works in Welsh prisons trying to secure housing for prison leavers, said the n

In [6]:
inputs = tokenizer(input, return_tensors="pt", max_length=512, truncation=True) # text to summarize
outputs = model.generate(inputs['input_ids'], max_length=50, output_scores=True, return_dict_in_generate=True) # generates summary in tokens
summary = tokenizer.decode(outputs[0], skip_special_tokens=True) # transforms tokens into words from the embedding table
print(summary)

['prison link Cymru says some ex-offenders were living rough for up to a year . the charity says it is cheaper than jailing homeless repeat offenders . the housing act in Wales was introduced in 2015 .']


In [7]:
outputs_beam = model.generate(inputs['input_ids'], max_length=50, num_beams=4) # Instead of picking the most likely next token, consider top 4 candidates at each step using cumulative probabilities

In [8]:
# Slightly different output here
summary_beam = tokenizer.decode(outputs_beam[0], skip_special_tokens=True)
summary_beam

'prison link Cymru says some ex-offenders were living rough for up to a year before finding suitable accommodation. change to the Housing Act in Wales removed the right for prison leavers to be given priority for accommodation '

In [15]:
# 1. Greedy (default)
greedy_outputs = model.generate(inputs['input_ids'], max_length=60)
greedy_summary = tokenizer.decode(greedy_outputs[0], skip_special_tokens=True)
greedy_summary

'prison link Cymru says some ex-offenders were living rough for up to a year. the charity says it is cheaper than jailing homeless repeat offenders. the housing act in Wales was introduced in 2015.'

In [12]:
# 2. Beam search
beam_outputs = model.generate(inputs['input_ids'], max_length=60, num_beams=4)
beam_summary = tokenizer.decode(beam_outputs[0], skip_special_tokens=True)
beam_summary

'prison link Cymru says some ex-offenders were living rough for up to a year before finding suitable accommodation. change to the Housing Act in Wales introduced in 2015 removed the right for prison leavers to be given priority for accommodation.'

In [13]:
# 3. Sampling with temperature (temperature controls how ample the differences between logits are: higher temp -> less differences)
low_temp_outputs = model.generate(inputs['input_ids'], max_length=60, do_sample=True, temperature=0.7)
low_temp_summary = tokenizer.decode(low_temp_outputs[0], skip_special_tokens=True)
low_temp_summary

'prison Link Cymru says it has 1,099 referrals in 2015-16. charity says a year spent making money before finding accommodation. charity says the need for accommodation is "chronic"'

In [14]:
# 4. Sampling with high temperature
high_temp_outputs = model.generate(inputs['input_ids'], max_length=60, do_sample=True, temperature=0.7)
high_temp_summary = tokenizer.decode(high_temp_outputs[0], skip_special_tokens=True)
high_temp_summary

'prison link Cymru says people getting help can help address housing problems. the charity says one-bedroom flats have a "chronic" need for accommodation. it said it would be cheaper than jailing homeless repeat offenders.'